# EII — Phase 2: Full Annual Time Series (1985–2024)

Runs EII and Area extraction for all available annual rasters using the
primary grid configuration (HEX-20, 20,000 ha hexagonal cells).

**Output:** two paired matrices of dimensions ~11,500 cells × N years:
- `eii_{year}`: Edge Interception Index per cell per year
- `area_{year}`: within-cell natural vegetation proportion per cell per year

**Key technical notes:**
- `nodata=255` is hardcoded for both metrics (raster metadata declares
  nodata=0, but 0 = non-natural vegetation, a valid value)
- EII uses `categorical=True` + `all_touched=True` on perimeter geometries
- Area uses `stats=['count','sum']` + `all_touched=False` on polygon interiors
- Checkpoint system saves one CSV per raster — safe to interrupt and resume

---
## 1. Configuration — edit only this cell

In [ ]:
# ============================================================
# CONFIGURATION — edit paths here, run everything else as-is
# ============================================================

# Folder containing ALL annual binary rasters (.tif)
# Expected naming pattern: reclass_YYYY.tif
RASTER_FOLDER = r"D:\Modelo_LAPIG\rasters_binarios"

# Primary grid shapefile (HEX-20, output of grid generation)
GRID_SHAPEFILE = r"D:\Modelo_LAPIG\grids\hex_20000ha.shp"

# Checkpoint folder — one CSV per raster, safe to interrupt and resume
CHECKPOINT_FOLDER = r"D:\Modelo_LAPIG\phase2_annual\checkpoints"

# Final consolidated output CSVs
EII_CSV_OUT  = r"D:\Modelo_LAPIG\phase2_annual\eii_HEX20_annual.csv"
AREA_CSV_OUT = r"D:\Modelo_LAPIG\phase2_annual\area_HEX20_annual.csv"
PAIRED_CSV_OUT = r"D:\Modelo_LAPIG\phase2_annual\eii_area_HEX20_annual.csv"

# Pixel value representing natural vegetation
VEGETATION_VALUE = 1

# Scenario label mapping
SCENARIO_MAP = {
    "reclass": "OBS",
    "obs":     "OBS",
    "tnc1":    "TNC1",
    "tnc2":    "TNC2",
    "bau":     "BAU",
    "gov":     "GOV",
}

---
## 2. Dependencies

In [ ]:
import importlib, subprocess, sys

for pkg in ["rasterstats", "geopandas", "rasterio", "pandas", "numpy", "matplotlib"]:
    if importlib.util.find_spec(pkg) is None:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
    else:
        print(f"OK {pkg}")

---
## 3. Utility functions

In [ ]:
import os
import re
import glob
import warnings
import time

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats

YEAR_RE = re.compile(r"(19|20)\d{2}")


def extract_year(filename: str) -> str:
    m = YEAR_RE.search(filename)
    if not m:
        raise ValueError(f"Year (YYYY) not found in: {filename}")
    return m.group(0)


def infer_scenario(filename: str, scenario_map: dict) -> str:
    lower = filename.lower()
    for key, label in scenario_map.items():
        if key.lower() in lower:
            return label
    return "OBS"


def ensure_id(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if "ID_UNICO" not in gdf.columns:
        gdf = gdf.copy()
        gdf["ID_UNICO"] = gdf.index.astype(int)
    return gdf


def to_perimeters(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Convert polygon grid to boundary lines (EII transects)."""
    gdf = gdf.copy()
    gdf["geometry"] = gdf.geometry.boundary
    gdf = gdf.explode(index_parts=False).reset_index(drop=True)
    return gdf


def reproject_if_needed(gdf: gpd.GeoDataFrame, raster_path: str) -> gpd.GeoDataFrame:
    with rasterio.open(raster_path) as src:
        crs_raster = src.crs
    if gdf.crs is None:
        raise ValueError("Grid has no CRS defined.")
    if crs_raster and gdf.crs != crs_raster:
        return gdf.to_crs(crs_raster)
    return gdf


print("Utility functions loaded.")

---
## 4. Extraction functions

Both functions use `nodata=255` regardless of raster metadata.
Raster metadata declares `nodata=0`, but 0 = non-natural vegetation (valid data).
True outside-domain pixels are encoded as 255.

In [ ]:
# Hardcoded nodata — do not change without verifying raster encoding
NODATA = 255


def extract_eii(gdf_perimeters, raster_path, checkpoint_folder):
    """
    Extract EII for one raster.
    Uses perimeter geometries as transects (all_touched=True).
    nodata=255: non-natural pixels (value 0) are correctly counted
    in the denominator.
    Returns DataFrame: ID_UNICO, eii_OBS_YYYY
    """
    base     = os.path.basename(raster_path)
    year     = extract_year(base)
    scenario = infer_scenario(base, SCENARIO_MAP)
    col      = f"eii_{scenario}_{year}"

    ckpt = os.path.join(checkpoint_folder, f"eii_{scenario}_{year}.csv")
    if os.path.exists(ckpt):
        return pd.read_csv(ckpt), year, "checkpoint"

    gdf_proj = reproject_if_needed(gdf_perimeters, raster_path)

    stats = zonal_stats(
        gdf_proj, raster_path,
        categorical=True,
        nodata=NODATA,
        all_touched=True,
    )

    ids    = gdf_perimeters["ID_UNICO"].values
    px_veg = np.array([s.get(VEGETATION_VALUE, 0) or 0 for s in stats], dtype=float)
    px_tot = np.array(
        [sum(v for k, v in s.items() if k != NODATA) for s in stats], dtype=float)

    with np.errstate(invalid="ignore", divide="ignore"):
        eii = np.where(px_tot > 0, px_veg / px_tot, np.nan)

    df = pd.DataFrame({"ID_UNICO": ids, col: np.round(eii, 4)})
    df.to_csv(ckpt, index=False)
    return df, year, "computed"


def extract_area(gdf_polygons, raster_path, checkpoint_folder):
    """
    Extract within-cell area metric for one raster.
    Uses polygon interiors (all_touched=False).
    nodata=255: non-natural pixels (value 0) are correctly counted
    in the denominator.
    Returns DataFrame: ID_UNICO, area_OBS_YYYY
    """
    base     = os.path.basename(raster_path)
    year     = extract_year(base)
    scenario = infer_scenario(base, SCENARIO_MAP)
    col      = f"area_{scenario}_{year}"

    ckpt = os.path.join(checkpoint_folder, f"area_{scenario}_{year}.csv")
    if os.path.exists(ckpt):
        return pd.read_csv(ckpt), year, "checkpoint"

    gdf_proj = reproject_if_needed(gdf_polygons, raster_path)

    stats = zonal_stats(
        gdf_proj, raster_path,
        stats=["count", "sum"],
        nodata=NODATA,
        all_touched=False,
    )

    ids    = gdf_polygons["ID_UNICO"].values
    px_tot = np.array([s["count"] or 0 for s in stats], dtype=float)
    px_veg = np.array([s["sum"]   or 0 for s in stats], dtype=float)

    with np.errstate(invalid="ignore", divide="ignore"):
        area = np.where(px_tot > 0, px_veg / px_tot, np.nan)

    df = pd.DataFrame({"ID_UNICO": ids, col: np.round(area, 4)})
    df.to_csv(ckpt, index=False)
    return df, year, "computed"


print("Extraction functions loaded.")

---
## 5. Load grid and list rasters

In [ ]:
os.makedirs(CHECKPOINT_FOLDER, exist_ok=True)

# Load HEX-20 grid
print("Loading HEX-20 grid...")
gdf = gpd.read_file(GRID_SHAPEFILE)
gdf = ensure_id(gdf)
print(f"  {len(gdf):,} cells | CRS: {gdf.crs}")

# Convert to perimeters for EII
print("Converting polygons to perimeter transects...")
gdf_perim = to_perimeters(gdf)
print(f"  {len(gdf_perim):,} perimeter features")

# List all annual rasters
rasters = sorted(glob.glob(os.path.join(RASTER_FOLDER, "*.tif")))
print(f"\n{len(rasters)} rasters found")

years = []
for r in rasters:
    try:
        y = extract_year(os.path.basename(r))
        years.append(y)
    except ValueError:
        print(f"  WARNING: no year found in {os.path.basename(r)} — skipping")

print(f"Years: {min(years)} to {max(years)} ({len(years)} rasters)")
if len(years) != len(set(years)):
    from collections import Counter
    dupes = [y for y, c in Counter(years).items() if c > 1]
    print(f"WARNING: duplicate years detected: {dupes}")
else:
    print("OK: all years are unique")

---
## 6. Run annual pipeline

Processes all rasters sequentially. Checkpoints allow safe interruption
and resumption — already-processed years are loaded from disk.

Estimated runtime: ~2–5 minutes per raster depending on hardware.
For 40 rasters: ~1.5–3.5 hours total. Run overnight if needed.

In [ ]:
df_eii  = gdf[["ID_UNICO"]].copy()
df_area = gdf[["ID_UNICO"]].copy()

n_total     = len(rasters)
n_computed  = 0
n_ckpt      = 0
t_start     = time.time()

print(f"Starting annual pipeline — {n_total} rasters")
print(f"Checkpoints: {CHECKPOINT_FOLDER}")
print("-" * 55)

for i, raster_path in enumerate(rasters, 1):
    base = os.path.basename(raster_path)
    year = extract_year(base)

    t0 = time.time()

    # EII
    df_e, _, status_e = extract_eii(gdf_perim, raster_path, CHECKPOINT_FOLDER)
    df_eii = df_eii.merge(df_e, on="ID_UNICO", how="left")

    # Area
    df_a, _, status_a = extract_area(gdf, raster_path, CHECKPOINT_FOLDER)
    df_area = df_area.merge(df_a, on="ID_UNICO", how="left")

    elapsed = time.time() - t0
    total_elapsed = time.time() - t_start

    if status_e == "computed":
        n_computed += 1
        remaining = (total_elapsed / i) * (n_total - i)
        print(f"  [{i:02d}/{n_total}] {year} | {elapsed:.0f}s | "
              f"est. remaining: {remaining/60:.0f} min")
    else:
        n_ckpt += 1
        print(f"  [{i:02d}/{n_total}] {year} | checkpoint")

print("-" * 55)
print(f"Done. Computed: {n_computed} | From checkpoint: {n_ckpt}")
print(f"Total time: {(time.time()-t_start)/60:.1f} min")
print(f"\nEII matrix:  {df_eii.shape}")
print(f"Area matrix: {df_area.shape}")

---
## 7. Export consolidated datasets

In [ ]:
os.makedirs(os.path.dirname(EII_CSV_OUT), exist_ok=True)

# Sort columns by year for readability
def sort_by_year(df):
    id_col   = ["ID_UNICO"]
    data_cols = sorted(
        [c for c in df.columns if c != "ID_UNICO"],
        key=lambda c: extract_year(c)
    )
    return df[id_col + data_cols]

df_eii  = sort_by_year(df_eii)
df_area = sort_by_year(df_area)

# Save separate matrices
df_eii.to_csv(EII_CSV_OUT,  index=False, encoding="utf-8-sig")
df_area.to_csv(AREA_CSV_OUT, index=False, encoding="utf-8-sig")
print(f"EII  matrix saved:  {EII_CSV_OUT}")
print(f"     Shape: {df_eii.shape} | Columns: {list(df_eii.columns[:4])} ...")
print(f"Area matrix saved:  {AREA_CSV_OUT}")
print(f"     Shape: {df_area.shape}")

# Save paired dataset (all columns together)
area_cols = [c for c in df_area.columns if c != "ID_UNICO"]
df_paired = df_eii.merge(df_area[["ID_UNICO"] + area_cols], on="ID_UNICO", how="left")
df_paired.to_csv(PAIRED_CSV_OUT, index=False, encoding="utf-8-sig")
print(f"Paired dataset saved: {PAIRED_CSV_OUT}")
print(f"     Shape: {df_paired.shape}")

---
## 8. Quality control

Checks for missing values, unexpected ranges, and temporal consistency.

In [ ]:
import matplotlib.pyplot as plt

eii_cols  = [c for c in df_eii.columns  if c != "ID_UNICO"]
area_cols = [c for c in df_area.columns if c != "ID_UNICO"]

# Missing value check
eii_nan_pct  = df_eii[eii_cols].isna().mean().mean() * 100
area_nan_pct = df_area[area_cols].isna().mean().mean() * 100
print(f"Missing values — EII: {eii_nan_pct:.2f}% | Area: {area_nan_pct:.2f}%")

# Range check
eii_min  = df_eii[eii_cols].min().min()
eii_max  = df_eii[eii_cols].max().max()
area_min = df_area[area_cols].min().min()
area_max = df_area[area_cols].max().max()
print(f"EII  range: [{eii_min:.4f}, {eii_max:.4f}]  (expected [0, 1])")
print(f"Area range: [{area_min:.4f}, {area_max:.4f}]  (expected [0, 1])")

if eii_max > 1.001 or eii_min < -0.001:
    print("WARNING: EII values outside [0,1] detected.")
if area_max > 1.001 or area_min < -0.001:
    print("WARNING: Area values outside [0,1] detected.")

# Temporal trend
eii_annual_mean  = df_eii[eii_cols].mean()
area_annual_mean = df_area[area_cols].mean()

years_num = [int(extract_year(c)) for c in eii_cols]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(years_num, eii_annual_mean.values,
        color="steelblue", linewidth=2, marker="o", markersize=4, label="EII")
ax.plot(years_num, area_annual_mean.values,
        color="forestgreen", linewidth=2, marker="s", markersize=4,
        linestyle="--", label="Area")
ax.set_xlabel("Year")
ax.set_ylabel("Mean value across cells")
ax.set_title("Annual mean EII and Area — HEX-20 (full domain)")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(min(years_num) - 1, max(years_num) + 1)
plt.tight_layout()
qc_path = os.path.join(os.path.dirname(EII_CSV_OUT), "qc_annual_trend.png")
plt.savefig(qc_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"QC trend plot saved: {qc_path}")

# Summary table
print("\nAnnual summary (first and last 5 years):")
summary = pd.DataFrame({
    "year":      years_num,
    "eii_mean":  eii_annual_mean.values.round(4),
    "area_mean": area_annual_mean.values.round(4),
    "eii_minus_area": (eii_annual_mean.values - area_annual_mean.values).round(4),
})
print(pd.concat([summary.head(5), summary.tail(5)]).to_string(index=False))

---
## 9. Annual decoupling overview

For each year, count cells in each of the four landscape states:
- **Coupled-High** (Area ≥ 0.5, EII ≥ 0.5): intact
- **Coupled-Low**  (Area < 0.5, EII < 0.5): degraded
- **Type I**       (Area ≥ 0.5, EII < 0.5): interior preserved, interface degraded
- **Type II**      (Area < 0.5, EII ≥ 0.5): interior lost, interface still connected

This gives the first answer to RQ1 across the full time series.

In [ ]:
THRESHOLD = 0.5
state_rows = []

for year_str, year_int in zip(eii_cols, years_num):
    eii_col_name  = year_str
    area_col_name = f"area_{year_str.split('_', 1)[1]}"

    if area_col_name not in df_area.columns:
        continue

    eii_v  = df_eii[eii_col_name].values
    area_v = df_area[area_col_name].values

    valid = np.isfinite(eii_v) & np.isfinite(area_v)
    e = eii_v[valid]
    a = area_v[valid]
    n = valid.sum()

    coupled_high = ((a >= THRESHOLD) & (e >= THRESHOLD)).sum()
    coupled_low  = ((a <  THRESHOLD) & (e <  THRESHOLD)).sum()
    type_i       = ((a >= THRESHOLD) & (e <  THRESHOLD)).sum()
    type_ii      = ((a <  THRESHOLD) & (e >= THRESHOLD)).sum()

    state_rows.append({
        "year":          year_int,
        "n_valid":       n,
        "coupled_high_n": coupled_high,
        "coupled_low_n":  coupled_low,
        "type_i_n":       type_i,
        "type_ii_n":      type_ii,
        "coupled_high_pct": round(100 * coupled_high / n, 2),
        "coupled_low_pct":  round(100 * coupled_low  / n, 2),
        "type_i_pct":       round(100 * type_i       / n, 2),
        "type_ii_pct":      round(100 * type_ii      / n, 2),
        "decoupled_pct":    round(100 * (type_i + type_ii) / n, 2),
    })

df_states = pd.DataFrame(state_rows)
states_path = os.path.join(os.path.dirname(EII_CSV_OUT), "annual_states.csv")
df_states.to_csv(states_path, index=False, encoding="utf-8-sig")

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.stackplot(
    df_states["year"],
    df_states["coupled_high_pct"],
    df_states["type_i_pct"],
    df_states["type_ii_pct"],
    df_states["coupled_low_pct"],
    labels=["Coupled-High", "Type I (high area, low EII)",
            "Type II (low area, high EII)", "Coupled-Low"],
    colors=["#2d6a4f", "#f4a261", "#457b9d", "#e63946"],
    alpha=0.85
)
ax.set_xlabel("Year")
ax.set_ylabel("% of cells")
ax.set_title("Annual landscape state distribution — HEX-20")
ax.legend(loc="lower left", fontsize=9)
ax.set_xlim(df_states["year"].min(), df_states["year"].max())
ax.set_ylim(0, 100)
plt.tight_layout()
states_fig = os.path.join(os.path.dirname(EII_CSV_OUT), "annual_states.png")
plt.savefig(states_fig, dpi=200, bbox_inches="tight")
plt.show()

print(f"\nState summary saved: {states_path}")
print("\nFirst and last 5 years:")
cols_show = ["year", "coupled_high_pct", "type_i_pct", "type_ii_pct",
             "coupled_low_pct", "decoupled_pct"]
print(pd.concat([df_states[cols_show].head(5),
                 df_states[cols_show].tail(5)]).to_string(index=False))

---
## 10. Decoupling dynamics — annual flux, spatial autocorrelation, and cumulative trace

Three complementary products that complete the answer to RQ1 and RQ2:

1. **Annual flux of decoupled states** — proportion of cells in each state per year,
   revealing transitions rather than just stock
2. **Moran's I global** — tests whether decoupled cells are spatially clustered
   (autocorrelated) or randomly distributed
3. **Cumulative decoupling trace** — map of cells that were ever in a decoupled
   state across the full time series

In [ ]:
import libpysal
from esda.moran import Moran
import warnings

# ── Setup ────────────────────────────────────────────────────────────────────
output_folder = os.path.dirname(EII_CSV_OUT)

# Rebuild eii_cols / years_num in case this cell is run independently
eii_cols  = [c for c in df_eii.columns  if c != "ID_UNICO"]
area_cols = [c for c in df_area.columns if c != "ID_UNICO"]
years_num = [int(extract_year(c)) for c in eii_cols]

# ── Product 1 — Annual flux of decoupled states ───────────────────────────
# (reuses df_states computed in cell 9 — run cell 9 first)
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Panel A: stacked area chart of all four states
ax = axes[0]
ax.stackplot(
    df_states["year"],
    df_states["coupled_high_pct"],
    df_states["type_i_pct"],
    df_states["type_ii_pct"],
    df_states["coupled_low_pct"],
    labels=["Coupled-High", "Type I (high area, low EII)",
             "Type II (low area, high EII)", "Coupled-Low"],
    colors=["#2d6a4f", "#f4a261", "#457b9d", "#e63946"],
    alpha=0.85
)
ax.set_ylabel("% of cells")
ax.set_title("Annual landscape state distribution")
ax.legend(loc="lower left", fontsize=9)
ax.set_ylim(0, 100)

# Panel B: decoupled states only (Type I + Type II), showing flux detail
ax2 = axes[1]
ax2.fill_between(df_states["year"], df_states["type_i_pct"],
                  color="#f4a261", alpha=0.7, label="Type I (high area, low EII)")
ax2.fill_between(df_states["year"], df_states["type_ii_pct"],
                  color="#457b9d", alpha=0.7, label="Type II (low area, high EII)")
ax2.plot(df_states["year"], df_states["decoupled_pct"],
          color="black", linewidth=1.5, label="Total decoupled")
ax2.set_xlabel("Year")
ax2.set_ylabel("% of cells")
ax2.set_title("Decoupled states — annual flux")
ax2.legend(fontsize=9)
ax2.set_xlim(min(years_num), max(years_num))

plt.tight_layout()
fig_path = os.path.join(output_folder, "annual_decoupling_flux.png")
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Figure saved: {fig_path}")

# ── Product 2 — Moran's I global per year ────────────────────────────────
# Requires: grid shapefile with geometry + libpysal + esda
# Install if needed:
import importlib, subprocess, sys
for pkg in ["libpysal", "esda"]:
    if importlib.util.find_spec(pkg) is None:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import libpysal
from esda.moran import Moran

print("\nBuilding spatial weights (queen contiguity)...")
# Load grid with geometry for spatial weights
gdf_w = gpd.read_file(GRID_SHAPEFILE)
gdf_w = ensure_id(gdf_w)
gdf_w = gdf_w.sort_values("ID_UNICO").reset_index(drop=True)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    w = libpysal.weights.Queen.from_dataframe(gdf_w, silence_warnings=True)
    w.transform = "r"  # row-standardize
print(f"  Weights built: {w.n} units, mean neighbors: {w.mean_neighbors:.1f}")

# Compute Moran's I for decoupled state (binary: 1=decoupled, 0=coupled)
moran_rows = []

for year_str, year_int in zip(eii_cols, years_num):
    area_col_name = f"area_{year_str.split('_', 1)[1]}"
    if area_col_name not in df_area.columns:
        continue

    eii_v  = df_eii[year_str].values
    area_v = df_area[area_col_name].values

    valid = np.isfinite(eii_v) & np.isfinite(area_v)

    # Binary decoupling variable (1 = any decoupled state)
    decoupled = np.where(
        valid,
        ((area_v >= THRESHOLD) & (eii_v < THRESHOLD) |
         (area_v <  THRESHOLD) & (eii_v >= THRESHOLD)).astype(float),
        np.nan
    )

    # Fill NaN with 0 for Moran computation (note in output)
    decoupled_filled = np.nan_to_num(decoupled, nan=0.0)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        mi = Moran(decoupled_filled, w, permutations=99)

    moran_rows.append({
        "year":    year_int,
        "moran_I": round(mi.I, 4),
        "p_value": round(mi.p_sim, 4),
        "z_score": round(mi.z_sim, 2),
        "significant": mi.p_sim < 0.05,
    })
    print(f"  {year_int}: I={mi.I:.4f}  p={mi.p_sim:.4f}  "
          f"({'significant' if mi.p_sim < 0.05 else 'n.s.'})")

df_moran = pd.DataFrame(moran_rows)
moran_path = os.path.join(output_folder, "moran_annual.csv")
df_moran.to_csv(moran_path, index=False, encoding="utf-8-sig")

# Plot Moran's I over time
fig, ax = plt.subplots(figsize=(12, 4))
colors = ["steelblue" if sig else "lightgray"
          for sig in df_moran["significant"]]
ax.bar(df_moran["year"], df_moran["moran_I"], color=colors, width=0.8)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Year")
ax.set_ylabel("Moran's I")
ax.set_title("Global spatial autocorrelation of decoupled states (Moran's I)\n"
             "Blue = significant (p<0.05), Gray = not significant")
ax.set_xlim(min(years_num) - 1, max(years_num) + 1)
plt.tight_layout()
moran_fig = os.path.join(output_folder, "moran_annual.png")
plt.savefig(moran_fig, dpi=200, bbox_inches="tight")
plt.show()
print(f"\nMoran results saved: {moran_path}")

# ── Product 3 — Cumulative decoupling trace ───────────────────────────────
# For each cell: was it EVER in a decoupled state across the full series?
print("\nComputing cumulative decoupling trace...")

ever_type_i  = np.zeros(len(df_eii), dtype=bool)
ever_type_ii = np.zeros(len(df_eii), dtype=bool)
n_years_decoupled = np.zeros(len(df_eii), dtype=int)

for year_str in eii_cols:
    area_col_name = f"area_{year_str.split('_', 1)[1]}"
    if area_col_name not in df_area.columns:
        continue

    eii_v  = df_eii[year_str].values
    area_v = df_area[area_col_name].values

    valid = np.isfinite(eii_v) & np.isfinite(area_v)
    type_i  = valid & (area_v >= THRESHOLD) & (eii_v < THRESHOLD)
    type_ii = valid & (area_v <  THRESHOLD) & (eii_v >= THRESHOLD)

    ever_type_i  |= type_i
    ever_type_ii |= type_ii
    n_years_decoupled += (type_i | type_ii).astype(int)

# Build GeoDataFrame with trace results
gdf_trace = gdf_w[["ID_UNICO", "geometry"]].copy()
gdf_trace["ever_type_i"]       = ever_type_i.astype(int)
gdf_trace["ever_type_ii"]      = ever_type_ii.astype(int)
gdf_trace["ever_decoupled"]    = (ever_type_i | ever_type_ii).astype(int)
gdf_trace["n_years_decoupled"] = n_years_decoupled
gdf_trace["pct_years_decoupled"] = np.round(
    n_years_decoupled / len(eii_cols) * 100, 2)

trace_shp = os.path.join(output_folder, "cumulative_decoupling_trace.shp")
gdf_trace.to_file(trace_shp)

# Summary
n_ever    = gdf_trace["ever_decoupled"].sum()
n_total_c = len(gdf_trace)
pct_ever  = 100 * n_ever / n_total_c
print(f"  Cells ever decoupled: {n_ever:,} / {n_total_c:,} ({pct_ever:.1f}%)")
print(f"  Cells ever Type I:    {ever_type_i.sum():,} ({100*ever_type_i.mean():.1f}%)")
print(f"  Cells ever Type II:   {ever_type_ii.sum():,} ({100*ever_type_ii.mean():.1f}%)")
print(f"  Mean years decoupled per cell: "
      f"{n_years_decoupled[n_years_decoupled>0].mean():.1f}")

# Map of cumulative trace
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Panel A: ever decoupled (binary)
gdf_trace.plot(
    column="ever_decoupled",
    ax=axes[0],
    cmap="RdYlGn_r",
    legend=True,
    legend_kwds={"label": "Ever decoupled (0=no, 1=yes)"},
    linewidth=0.1,
    edgecolor="none"
)
axes[0].set_title("Cells ever in decoupled state\n(any year, 1985–2024)",
                   fontsize=11)
axes[0].axis("off")

# Panel B: number of years in decoupled state
gdf_trace.plot(
    column="pct_years_decoupled",
    ax=axes[1],
    cmap="YlOrRd",
    legend=True,
    legend_kwds={"label": "% of years in decoupled state"},
    linewidth=0.1,
    edgecolor="none"
)
axes[1].set_title("Persistence of decoupling\n(% of years in series)",
                   fontsize=11)
axes[1].axis("off")

plt.suptitle("Cumulative decoupling trace — HEX-20, full time series",
              fontsize=12)
plt.tight_layout()
trace_fig = os.path.join(output_folder, "cumulative_decoupling_trace.png")
plt.savefig(trace_fig, dpi=200, bbox_inches="tight")
plt.show()
print(f"Trace shapefile saved: {trace_shp}")
print(f"Trace figure saved:    {trace_fig}")

# ── Product 4 — Delta (EII − Area) continuous divergence ────────────────────
# delta_i(t) = EII_i(t) − Area_i(t)
#   delta > 0: interface more connected than interior composition suggests
#   delta < 0: interface more degraded than interior composition suggests
#   delta = 0: composition and configuration aligned
# This is the continuous analogue of the 5x5 matrix — captures all divergence,
# including differences smaller than the 20% class threshold.

print("\nComputing delta (EII − Area) for all cells and years...")

delta_rows = []
delta_matrix = np.full((len(df_eii), len(eii_cols)), np.nan)

for j, (year_str, year_int) in enumerate(zip(eii_cols, years_num)):
    area_col_name = f"area_{year_str.split('_', 1)[1]}"
    if area_col_name not in df_area.columns:
        continue

    eii_v  = df_eii[year_str].values.astype(float)
    area_v = df_area[area_col_name].values.astype(float)
    delta  = eii_v - area_v
    delta_matrix[:, j] = delta

    valid = np.isfinite(delta)
    delta_rows.append({
        "year":          year_int,
        "delta_mean":    round(float(np.nanmean(delta)), 4),
        "delta_median":  round(float(np.nanmedian(delta)), 4),
        "delta_sd":      round(float(np.nanstd(delta)), 4),
        "delta_p10":     round(float(np.nanpercentile(delta, 10)), 4),
        "delta_p25":     round(float(np.nanpercentile(delta, 25)), 4),
        "delta_p75":     round(float(np.nanpercentile(delta, 75)), 4),
        "delta_p90":     round(float(np.nanpercentile(delta, 90)), 4),
        "pct_negative":  round(100 * (delta[valid] < 0).mean(), 2),
        "pct_positive":  round(100 * (delta[valid] > 0).mean(), 2),
        "pct_near_zero": round(100 * (np.abs(delta[valid]) < 0.05).mean(), 2),
    })

df_delta = pd.DataFrame(delta_rows)
delta_path = os.path.join(output_folder, "delta_annual_summary.csv")
df_delta.to_csv(delta_path, index=False, encoding="utf-8-sig")

# ── Figure A: mean delta and IQR over time ───────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax = axes[0]
ax.fill_between(df_delta["year"],
                 df_delta["delta_p10"], df_delta["delta_p90"],
                 alpha=0.15, color="steelblue", label="P10–P90")
ax.fill_between(df_delta["year"],
                 df_delta["delta_p25"], df_delta["delta_p75"],
                 alpha=0.30, color="steelblue", label="IQR (P25–P75)")
ax.plot(df_delta["year"], df_delta["delta_mean"],
         color="steelblue", linewidth=2, label="Mean delta")
ax.plot(df_delta["year"], df_delta["delta_median"],
         color="navy", linewidth=1.5, linestyle="--", label="Median delta")
ax.axhline(0, color="black", linewidth=0.8, linestyle="-")
ax.set_ylabel("EII − Area (delta)")
ax.set_title("Annual distribution of delta (EII − Area)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.fill_between(df_delta["year"], df_delta["pct_negative"],
                  alpha=0.6, color="#e63946",
                  label="% cells with delta < 0 (interface more degraded)")
ax2.fill_between(df_delta["year"], -df_delta["pct_positive"],
                  alpha=0.6, color="#2d6a4f",
                  label="% cells with delta > 0 (interface more connected)")
ax2.axhline(0, color="black", linewidth=0.8)
ax2.set_xlabel("Year")
ax2.set_ylabel("% of cells")
ax2.set_title("Direction of divergence — interface vs. interior")
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(min(years_num), max(years_num))

plt.tight_layout()
fig_path = os.path.join(output_folder, "delta_annual_trend.png")
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Delta trend figure saved: {fig_path}")

# ── Figure B: delta distribution (violin) for selected years ─────────────
# Select 5 representative years evenly spaced across the series
year_indices = np.linspace(0, len(eii_cols)-1, 5, dtype=int)
selected_years = [years_num[i] for i in year_indices]
selected_deltas = [delta_matrix[:, i] for i in year_indices]
selected_deltas = [d[np.isfinite(d)] for d in selected_deltas]

fig, ax = plt.subplots(figsize=(10, 5))
parts = ax.violinplot(
    selected_deltas,
    positions=range(len(selected_years)),
    showmedians=True,
    showextrema=False,
)
for pc in parts["bodies"]:
    pc.set_facecolor("steelblue")
    pc.set_alpha(0.6)
parts["cmedians"].set_color("navy")

ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xticks(range(len(selected_years)))
ax.set_xticklabels(selected_years)
ax.set_xlabel("Year")
ax.set_ylabel("EII − Area (delta)")
ax.set_title("Distribution of delta for selected years\n"
              "(shift below zero = interface degrading faster than interior)")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
violin_path = os.path.join(output_folder, "delta_violin_selected_years.png")
plt.savefig(violin_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Delta violin figure saved: {violin_path}")

# ── Summary ──────────────────────────────────────────────────────────────
print("\nDelta summary (first and last 5 years):")
cols_show = ["year", "delta_mean", "delta_median",
              "pct_negative", "pct_positive", "pct_near_zero"]
print(pd.concat([
    df_delta[cols_show].head(5),
    df_delta[cols_show].tail(5)
]).to_string(index=False))
print(f"\nDelta summary saved: {delta_path}")
